# Prepare Binned SBI Dataset For Cluster Runs

This notebook writes a self-contained `.npz` file for `run_sbi_for_cluster.pbs`. It prepares `theta`, `x`, `obs`, `ell`, and prior bounds with one consistent preprocessing chain:

1. choose source simulations: raw xgpaint/HalfDome spectra or an emulated SBI dataset,
2. choose the target binning: Planck 16-bin or SO 40-bin,
3. choose the observed `x_obs` path,
4. process `x_obs` with the same spectrum conversion, binning, and optional Gaussian beam as `x`,
5. save the prepared `.npz` and a JSON sidecar.

The saved vectors are already final. When submitting `run_sbi_for_cluster.pbs`, keep `SBI_GAUSSIAN_BEAM_MODE=off` unless you intentionally want the runner to apply an additional beam.


In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
if CWD.name == "SBI_analysis":
    PROJECT_ROOT = CWD.parent
elif (CWD / "SBI_analysis").is_dir():
    PROJECT_ROOT = CWD
elif (CWD.parent / "SBI_analysis").is_dir():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

SBI_DIR = PROJECT_ROOT / "SBI_analysis"
DATA_FOR_CLUSTER = SBI_DIR / "data_for_cluster"
DATA_FOR_CLUSTER.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


## User Configuration

Edit only this cell for normal use. The defaults produce an SO 40-bin emulated, no-beam dataset using the existing binned SO observation.


In [ ]:
# Source simulations: "emulated" or "xgpaint".
SOURCE_MODE = "emulated"

# Target binning: "SO" or "PLANCK".
BINNING_CONFIG = "SO"

DEFAULT_EMULATED_DATASETS = {
    "SO": PROJECT_ROOT / "emulator_tSZ" / "outputs" / "binned_40" / "sbi_100k_uniform_prior_emulated_log10_dl.npz",
    "PLANCK": PROJECT_ROOT / "emulator_tSZ" / "outputs" / "binned_16e3_16" / "sbi16e3_bin16_dataset_100_000.npz",
}
DEFAULT_XGPAINT_DATASET = PROJECT_ROOT / "HPC_output" / "HalfDome" / "ydata_12274" / "sbi_battaglia_y100_12274.npz"

SOURCE_DATASET_PATH = DEFAULT_EMULATED_DATASETS[BINNING_CONFIG] if SOURCE_MODE == "emulated" else DEFAULT_XGPAINT_DATASET

# For xgpaint/raw NPZs, choose which C_l matrix to use if several are present.
# Common values: "cl_mean", "cl_y100", "cl_concat".
XGPAINT_CL_KEY = "cl_mean"

DEFAULT_OBS_PATHS = {
    "SO": PROJECT_ROOT / "emulator_tSZ" / "outputs" / "binned_40" / "x_obs_log10_dl.npy",
    "PLANCK": PROJECT_ROOT / "HPC_output" / "HalfDome" / "tSZ_HalfDome_fiducial_nside4096_cluster.npy",
}
OBS_PATH = DEFAULT_OBS_PATHS[BINNING_CONFIG]

# Observation input kind: "auto", "cl", "dl", or "log10_dl".
# Use "auto" for the default paths above.
OBS_INPUT_KIND = "auto"

# Apply the same Gaussian beam to both prepared x and obs after binning.
APPLY_GAUSSIAN_BEAM = False
GAUSSIAN_BEAM_FWHM_ARCMIN = 2.0

# If an emulated source already has exactly the requested number of bins, trust it as pre-binned.
ASSUME_PREBINNED_IF_DIM_MATCH = True

# Binning statistics. Keep these unless you are intentionally changing the setup.
PLANCK_STATISTIC = "mean"
PLANCK_WEIGHTING = "uniform"
SO_STATISTIC = "mean"
SO_WEIGHTING = "2ell_plus_1"

# Processing and output.
FLOOR_DL = 1.0e-40
CHUNK_ROWS = 2048
OUTPUT_TAG = f"{SOURCE_MODE}_{BINNING_CONFIG.lower()}_binned{'_beam' if APPLY_GAUSSIAN_BEAM else '_no_beam'}"
OUTPUT_PATH = DATA_FOR_CLUSTER / f"{OUTPUT_TAG}_sbi_run.npz"
WRITE_OUTPUT = True

print("source mode:", SOURCE_MODE)
print("binning:", BINNING_CONFIG)
print("source dataset:", SOURCE_DATASET_PATH)
print("observation:", OBS_PATH)
print("output:", OUTPUT_PATH)


In [ ]:
SOBOL_PRIOR_BOUNDS = {
    "P0": [1.832524, 34.341221],
    "xc": [0.150011, 0.844503],
    "beta": [3.480627, 5.216611],
    "alpha_m_P0": [0.000312, 0.292251],
    "alpha_m_xc": [-0.099718, 0.099795],
    "alpha_m_beta": [-0.019935, 0.099767],
    "alpha_z_P0": [-1.363457, -0.228839],
    "alpha_z_xc": [0.147393, 1.314474],
    "alpha_z_beta": [0.083808, 0.745884],
}

DEFAULT_PARAM_NAMES = np.asarray(list(SOBOL_PRIOR_BOUNDS.keys()))
DEFAULT_PRIOR_LOW = np.asarray([SOBOL_PRIOR_BOUNDS[name][0] for name in DEFAULT_PARAM_NAMES], dtype=np.float32)
DEFAULT_PRIOR_HIGH = np.asarray([SOBOL_PRIOR_BOUNDS[name][1] for name in DEFAULT_PARAM_NAMES], dtype=np.float32)


def resolve_path(path: str | Path) -> Path:
    path = Path(path).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path


def npz_scalar_to_str(data: np.lib.npyio.NpzFile, key: str, default: str = "") -> str:
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.shape == ():
        return str(value.item())
    if value.size == 1:
        return str(value.reshape(-1)[0])
    return default


def npz_json_dict(data: np.lib.npyio.NpzFile, key: str) -> dict[str, Any]:
    raw = npz_scalar_to_str(data, key, default="")
    if not raw:
        return {}
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return {}
    return parsed if isinstance(parsed, dict) else {}


def jsonable(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return repr(value)


## Binning Definitions


In [ ]:
@dataclass(frozen=True)
class BinningSpec:
    name: str
    bin_min: np.ndarray
    bin_max: np.ndarray
    ell: np.ndarray
    statistic: str
    weighting: str
    edge_mode: str

    @property
    def n_bins(self) -> int:
        return int(self.ell.size)


def bin_weights(ell_values: np.ndarray, weighting: str) -> np.ndarray:
    ell_values = np.asarray(ell_values, dtype=np.float64)
    weighting = str(weighting or "uniform").lower()
    if weighting in {"uniform", "none", "flat"}:
        return np.ones_like(ell_values, dtype=np.float64)
    if weighting == "ell":
        return ell_values.astype(np.float64)
    if weighting in {"2ell_plus_1", "modes", "mode_count"}:
        return 2.0 * ell_values.astype(np.float64) + 1.0
    raise ValueError(f"Unsupported bin weighting {weighting!r}")


def weighted_center(lo: int, hi: int, weighting: str) -> float:
    ell_values = np.arange(int(lo), int(hi) + 1, dtype=np.float64)
    return float(np.average(ell_values, weights=bin_weights(ell_values, weighting)))


def make_planck_spec() -> BinningSpec:
    bin_min = np.asarray([21, 27, 35, 46, 60, 78, 102, 133, 173, 224, 292, 380, 494, 642, 835, 1085], dtype=np.int64)
    bin_max = np.asarray([26, 34, 45, 59, 77, 101, 132, 172, 223, 291, 379, 493, 641, 834, 1084, 1410], dtype=np.int64)
    ell = np.asarray([weighted_center(lo, hi, PLANCK_WEIGHTING) for lo, hi in zip(bin_min, bin_max)], dtype=np.float32)
    return BinningSpec("PLANCK", bin_min, bin_max, ell, PLANCK_STATISTIC, PLANCK_WEIGHTING, "inclusive")


def make_so_spec() -> BinningSpec:
    edges = np.r_[np.arange(80, 7881, 200), 7979].astype(np.int64)
    bin_min = edges[:-1].copy()
    bin_max = edges[1:].copy()
    bin_max[:-1] -= 1
    ell = np.asarray([weighted_center(lo, hi, SO_WEIGHTING) for lo, hi in zip(bin_min, bin_max)], dtype=np.float32)
    return BinningSpec("SO", bin_min, bin_max, ell, SO_STATISTIC, SO_WEIGHTING, "left_closed_last_inclusive")


def make_binning_spec(name: str) -> BinningSpec:
    name = str(name).strip().upper()
    if name == "PLANCK":
        return make_planck_spec()
    if name == "SO":
        return make_so_spec()
    raise ValueError("BINNING_CONFIG must be 'PLANCK' or 'SO'")


BINNING = make_binning_spec(BINNING_CONFIG)
print(BINNING)
print("bin count:", BINNING.n_bins)
print("ell range:", float(BINNING.ell[0]), float(BINNING.ell[-1]))


In [ ]:
def bin_member_indices(source_ell: np.ndarray, spec: BinningSpec) -> list[np.ndarray]:
    source_ell = np.asarray(source_ell, dtype=np.float64).reshape(-1)
    members: list[np.ndarray] = []
    for lo, hi in zip(spec.bin_min, spec.bin_max):
        idx = np.flatnonzero((source_ell >= float(lo)) & (source_ell <= float(hi)))
        if idx.size == 0:
            raise ValueError(
                f"No source ell values found for {spec.name} bin {lo}-{hi}. "
                f"Source ell range is {source_ell.min()}-{source_ell.max()}."
            )
        members.append(idx)
    return members


def bin_last_axis_log10(values: np.ndarray, source_ell: np.ndarray, spec: BinningSpec) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    source_ell = np.asarray(source_ell, dtype=np.float64).reshape(-1)
    if values.shape[-1] != source_ell.size:
        raise ValueError(f"values last axis {values.shape[-1]} does not match source ell length {source_ell.size}")

    statistic = str(spec.statistic).lower()
    pieces = []
    for idx in bin_member_indices(source_ell, spec):
        part = values[..., idx]
        if statistic == "mean":
            weights = bin_weights(source_ell[idx], spec.weighting)
            pieces.append(np.average(part, axis=-1, weights=weights))
        elif statistic == "median":
            pieces.append(np.median(part, axis=-1))
        else:
            raise ValueError(f"Unsupported bin statistic {spec.statistic!r}")
    return np.ascontiguousarray(np.stack(pieces, axis=-1), dtype=np.float32)


def cl_to_log10_dl(cl: np.ndarray, ell: np.ndarray, floor_dl: float = FLOOR_DL) -> np.ndarray:
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    factor = ell * (ell + 1.0) / (2.0 * math.pi)
    return np.log10(np.maximum(np.asarray(cl, dtype=np.float64) * factor, float(floor_dl)))


def dl_to_log10_dl(dl: np.ndarray, floor_dl: float = FLOOR_DL) -> np.ndarray:
    return np.log10(np.maximum(np.asarray(dl, dtype=np.float64), float(floor_dl)))


def values_to_log10_dl(values: np.ndarray, ell: np.ndarray, kind: str) -> np.ndarray:
    kind = str(kind).lower()
    if kind == "log10_dl":
        return np.asarray(values, dtype=np.float64)
    if kind == "dl":
        return dl_to_log10_dl(values)
    if kind == "cl":
        return cl_to_log10_dl(values, ell)
    raise ValueError("kind must be cl, dl, or log10_dl")


def gaussian_beam_window(ell: np.ndarray, fwhm_arcmin: float) -> np.ndarray:
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    fwhm_arcmin = float(fwhm_arcmin)
    if fwhm_arcmin < 0.0:
        raise ValueError("GAUSSIAN_BEAM_FWHM_ARCMIN must be non-negative")
    if fwhm_arcmin == 0.0:
        return np.ones_like(ell, dtype=np.float64)
    fwhm_rad = np.deg2rad(fwhm_arcmin / 60.0)
    sigma_rad = fwhm_rad / np.sqrt(8.0 * np.log(2.0))
    return np.exp(-0.5 * ell * (ell + 1.0) * sigma_rad**2)


def apply_gaussian_beam_to_log10_dl(values: np.ndarray, ell: np.ndarray, fwhm_arcmin: float) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    ell = np.asarray(ell, dtype=np.float64).reshape(-1)
    if values.shape[-1] != ell.size:
        raise ValueError(f"Beam ell length {ell.size} does not match values last dimension {values.shape[-1]}")
    beam = gaussian_beam_window(ell, fwhm_arcmin)
    beam_log_factor = np.log10(np.maximum(beam**2, FLOOR_DL)).astype(np.float32)
    return np.ascontiguousarray(values + beam_log_factor, dtype=np.float32)


## Load And Process Simulation Dataset


In [ ]:
def infer_matrix_kind(key: str, target_kind: str, values: np.ndarray) -> str:
    key_l = str(key).lower()
    target_l = str(target_kind).lower()
    if "log10" in key_l or "log10" in target_l:
        return "log10_dl"
    if key_l.startswith("cl") or "c_l" in key_l:
        return "cl"
    if "dl" in key_l or "d_l" in key_l:
        return "dl"
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size:
        med = float(np.nanmedian(finite[: min(finite.size, 100000)]))
        if med < 0.0:
            return "log10_dl"
        if abs(med) > 1.0e-15:
            return "dl"
    return "cl"


def choose_source_matrix(data: np.lib.npyio.NpzFile, source_mode: str, xgpaint_cl_key: str) -> tuple[str, str]:
    source_mode = str(source_mode).lower()
    if source_mode == "xgpaint":
        if xgpaint_cl_key not in data.files:
            cl_keys = [key for key in data.files if key.startswith("cl")]
            raise KeyError(f"{xgpaint_cl_key!r} not found. Available C_l keys: {cl_keys}")
        return xgpaint_cl_key, "cl"

    for key in ("x_log10_dl", "x_binned", "x"):
        if key in data.files:
            target_kind = npz_scalar_to_str(data, "target_kind", default="")
            return key, infer_matrix_kind(key, target_kind, data[key])
    for key in ("cl_mean", "cl_y100", "cl_concat"):
        if key in data.files:
            return key, "cl"
    raise KeyError(f"Could not choose an x matrix. Available keys: {data.files}")


def source_param_names(data: np.lib.npyio.NpzFile, theta: np.ndarray) -> np.ndarray:
    for key in ("theta_columns", "param_names", "x_columns"):
        if key in data.files:
            return np.asarray(data[key]).astype(str)
    if theta.shape[1] == DEFAULT_PARAM_NAMES.size:
        return DEFAULT_PARAM_NAMES.copy()
    return np.asarray([f"theta_{idx}" for idx in range(theta.shape[1])])


def source_prior_bounds(data: np.lib.npyio.NpzFile, param_names: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if "prior_low" in data.files and "prior_high" in data.files:
        return np.asarray(data["prior_low"], dtype=np.float32), np.asarray(data["prior_high"], dtype=np.float32)
    lows = []
    highs = []
    for name in param_names:
        if str(name) not in SOBOL_PRIOR_BOUNDS:
            raise KeyError(f"Missing prior bounds for parameter {name!r}")
        lo, hi = SOBOL_PRIOR_BOUNDS[str(name)]
        lows.append(lo)
        highs.append(hi)
    return np.asarray(lows, dtype=np.float32), np.asarray(highs, dtype=np.float32)


def direct_grid_match(source_ell: np.ndarray, spec: BinningSpec, n_columns: int) -> bool:
    if int(n_columns) != spec.n_bins:
        return False
    source_ell = np.asarray(source_ell, dtype=np.float64).reshape(-1)
    if source_ell.size != spec.n_bins:
        return bool(ASSUME_PREBINNED_IF_DIM_MATCH)
    if np.allclose(source_ell, spec.ell, rtol=1.0e-4, atol=1.0e-3):
        return True
    return bool(ASSUME_PREBINNED_IF_DIM_MATCH)


def process_matrix_to_target_bins(matrix: np.ndarray, source_ell: np.ndarray, matrix_kind: str, spec: BinningSpec) -> np.ndarray:
    matrix = np.asarray(matrix)
    source_ell = np.asarray(source_ell, dtype=np.float64).reshape(-1)
    if matrix.ndim != 2:
        raise ValueError(f"simulation matrix must be 2D, got {matrix.shape}")
    if direct_grid_match(source_ell, spec, matrix.shape[1]) and matrix_kind == "log10_dl":
        return np.ascontiguousarray(matrix, dtype=np.float32)
    if matrix.shape[1] != source_ell.size:
        raise ValueError(f"matrix columns {matrix.shape[1]} do not match ell length {source_ell.size}")

    out = np.empty((matrix.shape[0], spec.n_bins), dtype=np.float32)
    for start in range(0, matrix.shape[0], int(CHUNK_ROWS)):
        stop = min(start + int(CHUNK_ROWS), matrix.shape[0])
        log10_dl = values_to_log10_dl(matrix[start:stop], source_ell, matrix_kind)
        out[start:stop] = bin_last_axis_log10(log10_dl, source_ell, spec)
    return np.ascontiguousarray(out, dtype=np.float32)


def load_and_process_simulations(path: str | Path, source_mode: str, spec: BinningSpec) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
    path = resolve_path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing source dataset: {path}")

    with np.load(path, allow_pickle=True) as data:
        if "theta" not in data.files:
            raise KeyError(f"{path} is missing theta")
        if "ell" not in data.files:
            raise KeyError(f"{path} is missing ell")

        theta = np.ascontiguousarray(data["theta"], dtype=np.float32)
        param_names = source_param_names(data, theta)
        prior_low, prior_high = source_prior_bounds(data, param_names)
        source_ell = np.asarray(data["ell"], dtype=np.float64).reshape(-1)
        x_key, matrix_kind = choose_source_matrix(data, source_mode, XGPAINT_CL_KEY)
        matrix = data[x_key]
        binning_json = npz_json_dict(data, "binning_json")

        x = process_matrix_to_target_bins(matrix, source_ell, matrix_kind, spec)

    if theta.shape[0] != x.shape[0]:
        raise ValueError(f"theta rows {theta.shape[0]} do not match x rows {x.shape[0]}")

    payload = {
        "theta": theta,
        "x": x,
        "ell": np.ascontiguousarray(spec.ell, dtype=np.float32),
        "prior_low": np.ascontiguousarray(prior_low, dtype=np.float32),
        "prior_high": np.ascontiguousarray(prior_high, dtype=np.float32),
        "param_names": np.asarray(param_names).astype(str),
    }
    summary = {
        "source_dataset": str(path),
        "source_mode": source_mode,
        "source_x_key": x_key,
        "source_matrix_kind": matrix_kind,
        "source_ell_min": float(np.nanmin(source_ell)),
        "source_ell_max": float(np.nanmax(source_ell)),
        "source_binning_json": binning_json,
        "n_rows": int(theta.shape[0]),
        "theta_dim": int(theta.shape[1]),
        "x_dim": int(x.shape[1]),
    }
    return payload, summary


prepared, simulation_summary = load_and_process_simulations(SOURCE_DATASET_PATH, SOURCE_MODE, BINNING)
print("theta:", prepared["theta"].shape)
print("x:", prepared["x"].shape)
print("ell:", prepared["ell"].shape, prepared["ell"][0], prepared["ell"][-1])


## Load And Process `x_obs` With The Same Setup


In [ ]:
def normalize_name(name: str) -> str:
    import re
    normalized = name.strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)
    normalized = re.sub(r"_+", "_", normalized)
    return normalized.strip("_")


def read_spectrum_file(path: str | Path) -> tuple[np.ndarray, np.ndarray | None]:
    path = resolve_path(path)
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.asarray(np.load(path), dtype=np.float64).squeeze(), None
    if suffix == ".npz":
        with np.load(path, allow_pickle=True) as data:
            ell = np.asarray(data["ell"], dtype=np.float64).squeeze() if "ell" in data.files else None
            for key in ("obs", "x_obs", "x_obs_log10_dl", "x_observed_log10", "log10_dl", "dl", "d_l", "cl", "c_l", "profile", "data", "x"):
                if key in data.files:
                    return np.asarray(data[key], dtype=np.float64).squeeze(), ell
            if len(data.files) == 1:
                return np.asarray(data[data.files[0]], dtype=np.float64).squeeze(), ell
            raise ValueError(f"Could not choose observed spectrum inside {path}; keys={data.files}")
    if suffix in (".csv", ".txt", ".dat"):
        raw = np.loadtxt(path, delimiter="," if suffix == ".csv" else None)
        raw = np.asarray(raw, dtype=np.float64)
        if raw.ndim == 2 and raw.shape[1] >= 2:
            return raw[:, 1], raw[:, 0]
        return raw.squeeze(), None
    if suffix in (".fits", ".fit", ".fts"):
        try:
            from astropy.io import fits
        except ImportError as exc:
            raise ImportError("Reading FITS observations requires astropy in this kernel") from exc
        profile_candidates = []
        ell_candidates = []
        preferred = {"c_l", "cl", "dl", "d_l", "power", "profile"}
        ell_names = {"ell", "l", "multipole"}
        skip = {"index", "row", "pixel"}
        with fits.open(path, memmap=False) as hdul:
            for hdu_index, hdu in enumerate(hdul):
                data = hdu.data
                if data is None:
                    continue
                if getattr(data, "dtype", None) is not None and data.dtype.fields:
                    for field in data.dtype.names or ():
                        arr = np.asarray(data[field]).squeeze()
                        if arr.size < 2 or not np.issubdtype(arr.dtype, np.number):
                            continue
                        key = normalize_name(field)
                        arr = np.asarray(arr, dtype=np.float64).reshape(-1)
                        if key in ell_names:
                            ell_candidates.append(arr)
                        elif key not in skip:
                            priority = 0 if key in preferred or "cl" in key or "power" in key else 1
                            profile_candidates.append((priority, hdu_index, arr))
                else:
                    arr = np.asarray(data).squeeze()
                    if arr.size >= 2 and np.issubdtype(arr.dtype, np.number):
                        profile_candidates.append((2, hdu_index, np.asarray(arr, dtype=np.float64).reshape(-1)))
        if not profile_candidates:
            raise ValueError(f"No spectrum-like numeric data found in {path}")
        profile_candidates.sort(key=lambda item: (item[0], -item[2].size))
        profile = profile_candidates[0][2]
        ell = ell_candidates[0] if ell_candidates and ell_candidates[0].size == profile.size else None
        return profile, ell
    raise ValueError(f"Unsupported observed spectrum extension: {path}")


def infer_observed_kind(path: str | Path, spectrum: np.ndarray, ell: np.ndarray) -> str:
    name = normalize_name(Path(path).stem)
    tokens = set(name.split("_"))
    if "log10" in tokens and ({"dl", "d", "ell"} & tokens):
        return "log10_dl"
    if "dl" in tokens or "dell" in tokens or "d_ell" in name:
        return "dl"
    if "cl" in tokens or "cell" in tokens or "c_ell" in name:
        return "cl"
    if Path(path).suffix.lower() in (".fits", ".fit", ".fts"):
        return "cl"

    good = np.isfinite(spectrum) & np.isfinite(ell)
    finite = spectrum[good] if np.any(good) else spectrum[np.isfinite(spectrum)]
    if finite.size == 0:
        raise ValueError(f"Observed spectrum {path} has no finite values")
    med_signed = float(np.nanmedian(finite))
    med_abs = float(np.nanmedian(np.abs(finite)))
    if med_signed < 0.0:
        return "log10_dl"
    return "dl" if med_abs > 1.0e-15 else "cl"


def process_observation_to_target_bins(path: str | Path, input_kind: str, spec: BinningSpec) -> tuple[np.ndarray, dict[str, Any]]:
    path = resolve_path(path)
    spectrum, obs_ell = read_spectrum_file(path)
    spectrum = np.asarray(spectrum, dtype=np.float64).squeeze()
    if spectrum.ndim != 1:
        raise ValueError(f"Observed spectrum must be 1D after loading, got {spectrum.shape}")
    if obs_ell is None:
        obs_ell = np.arange(spectrum.size, dtype=np.float64)
    else:
        obs_ell = np.asarray(obs_ell, dtype=np.float64).reshape(-1)
    if obs_ell.size != spectrum.size:
        raise ValueError(f"Observed ell length {obs_ell.size} does not match spectrum length {spectrum.size}")

    kind = infer_observed_kind(path, spectrum, obs_ell) if str(input_kind).lower() == "auto" else str(input_kind).lower()

    if spectrum.size == spec.n_bins and kind == "log10_dl" and ASSUME_PREBINNED_IF_DIM_MATCH:
        obs = np.ascontiguousarray(spectrum, dtype=np.float32)
        direct = True
    elif spectrum.size == spec.n_bins and ASSUME_PREBINNED_IF_DIM_MATCH:
        obs = np.ascontiguousarray(values_to_log10_dl(spectrum, spec.ell, kind), dtype=np.float32)
        direct = True
    else:
        log10_dl = values_to_log10_dl(spectrum, obs_ell, kind)
        obs = bin_last_axis_log10(log10_dl.reshape(1, -1), obs_ell, spec).reshape(-1)
        direct = False

    summary = {
        "obs_path": str(path),
        "obs_input_kind_requested": input_kind,
        "obs_input_kind_used": kind,
        "obs_direct_prebinned": bool(direct),
        "obs_source_length": int(spectrum.size),
        "obs_source_ell_min": float(np.nanmin(obs_ell)),
        "obs_source_ell_max": float(np.nanmax(obs_ell)),
    }
    return obs, summary


obs, obs_summary = process_observation_to_target_bins(OBS_PATH, OBS_INPUT_KIND, BINNING)
prepared["obs"] = np.ascontiguousarray(obs, dtype=np.float32)
print("obs:", prepared["obs"].shape)
print(obs_summary)


## Optional Gaussian Beam

If enabled, the same Gaussian beam factor is added to both simulation vectors and the observation in `log10(D_l)` space. The cluster runner should then be submitted with beam mode off to avoid double application.


In [ ]:
if APPLY_GAUSSIAN_BEAM:
    prepared["x"] = apply_gaussian_beam_to_log10_dl(prepared["x"], prepared["ell"], GAUSSIAN_BEAM_FWHM_ARCMIN)
    prepared["obs"] = apply_gaussian_beam_to_log10_dl(prepared["obs"], prepared["ell"], GAUSSIAN_BEAM_FWHM_ARCMIN)

beam_summary = {
    "applied_in_preparation": bool(APPLY_GAUSSIAN_BEAM),
    "fwhm_arcmin": float(GAUSSIAN_BEAM_FWHM_ARCMIN if APPLY_GAUSSIAN_BEAM else 0.0),
    "runner_should_use_mode": "off",
    "runner_should_use_fwhm_arcmin": 0.0,
}
beam_summary


## Validate And Preview


In [ ]:
if prepared["theta"].ndim != 2:
    raise ValueError(f"theta must be 2D, got {prepared['theta'].shape}")
if prepared["x"].ndim != 2:
    raise ValueError(f"x must be 2D, got {prepared['x'].shape}")
if prepared["theta"].shape[0] != prepared["x"].shape[0]:
    raise ValueError("theta and x row counts do not match")
if prepared["obs"].shape != (prepared["x"].shape[1],):
    raise ValueError(f"obs shape {prepared['obs'].shape} does not match x dimension {prepared['x'].shape[1]}")
for key in ("theta", "x", "obs", "ell", "prior_low", "prior_high"):
    if not np.all(np.isfinite(prepared[key])):
        raise ValueError(f"{key} contains non-finite values")

print("n rows:", prepared["theta"].shape[0])
print("theta dim:", prepared["theta"].shape[1])
print("x dim:", prepared["x"].shape[1])
print("x min/max:", float(np.min(prepared["x"])), float(np.max(prepared["x"])))
print("obs min/max:", float(np.min(prepared["obs"])), float(np.max(prepared["obs"])))

plt.figure(figsize=(7, 4))
plt.plot(prepared["ell"], prepared["obs"], label="processed obs", lw=2.0)
plt.plot(prepared["ell"], prepared["x"][0], label="first simulation", alpha=0.8)
plt.plot(prepared["ell"], np.median(prepared["x"][: min(1024, prepared["x"].shape[0])], axis=0), label="median first rows", alpha=0.8)
plt.xscale("log")
plt.xlabel("ell")
plt.ylabel("log10(D_l)" + (" with beam" if APPLY_GAUSSIAN_BEAM else ""))
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## Save Cluster Dataset


In [ ]:
binning_summary = {
    "name": BINNING.name,
    "n_bins": BINNING.n_bins,
    "statistic": BINNING.statistic,
    "weighting": BINNING.weighting,
    "bin_ell_min": BINNING.bin_min.astype(int).tolist(),
    "bin_ell_max": BINNING.bin_max.astype(int).tolist(),
}

summary = {
    **simulation_summary,
    **obs_summary,
    "binning": binning_summary,
    "gaussian_beam": beam_summary,
    "output_path": str(resolve_path(OUTPUT_PATH)),
}

save_payload = {
    **prepared,
    "bin_counts": np.ascontiguousarray(BINNING.bin_max - BINNING.bin_min + 1, dtype=np.int64),
    "bin_ell_min": np.ascontiguousarray(BINNING.bin_min, dtype=np.float32),
    "bin_ell_max": np.ascontiguousarray(BINNING.bin_max, dtype=np.float32),
    "source_dataset": np.asarray(str(resolve_path(SOURCE_DATASET_PATH))),
    "source_mode": np.asarray(str(SOURCE_MODE)),
    "source_x_key": np.asarray(str(simulation_summary["source_x_key"])),
    "obs_path": np.asarray(str(resolve_path(OBS_PATH))),
    "binning_json": np.asarray(json.dumps(binning_summary, sort_keys=True)),
    "metadata_json": np.asarray(json.dumps(summary, sort_keys=True)),
    "gaussian_beam_applied": np.asarray(bool(APPLY_GAUSSIAN_BEAM)),
    "gaussian_beam_fwhm_arcmin": np.asarray(float(GAUSSIAN_BEAM_FWHM_ARCMIN if APPLY_GAUSSIAN_BEAM else 0.0)),
}

output_path = resolve_path(OUTPUT_PATH)
summary_path = output_path.with_suffix(".json")

if WRITE_OUTPUT:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_path, **save_payload)
    with summary_path.open("w", encoding="utf-8") as handle:
        json.dump(jsonable(summary), handle, indent=2, sort_keys=True)
    print("wrote", output_path)
    print("wrote", summary_path)
else:
    print("WRITE_OUTPUT is False; not writing files.")

print("Use this for PBS:")
print(f"qsub -v PREPARED_DATASET_PATH={output_path},SBI_GAUSSIAN_BEAM_MODE=off,SBI_GAUSSIAN_BEAM_FWHM_ARCMIN=0.0 SBI_analysis/run_sbi_for_cluster.pbs")
